In [1]:
#Task 1
# Veriyi hazırla. FineWeb-Edu'dan küçük bir parça (birkaç on milyon token
# yeterli) ya da kendi Türkçe metnini GPT-2 tokenizer'ıyla token'la.
# Token'ları dosyalara böl ve data loader'ını bu dosyalardan okuyacak hâle
# getir. Bir dosyayı doğrulama için ayır. (3:10:21 - 3:23:10)

In [2]:
#imports
import tiktoken
import torch
import numpy as np
import os
from datasets import load_dataset
from tqdm import tqdm
import multiprocessing as mp
from google.colab import drive
B, T = 16, 1024
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
def load_tokens(filename):
    npt = np.load(filename)
    ptt = torch.tensor(npt, dtype=torch.long)
    return ptt

In [4]:
#how to load data to model
class DataLoaderLite:
    def __init__(self, B, T, split):
        self.B = B
        self.T = T
        assert split in {'train', 'val'}


        data_root = "/content/drive/MyDrive/yz50/edu_fineweb10B"
        shards = os.listdir(data_root)
        shards = [s for s in shards if split in s]
        shards = sorted(shards)
        shards = [os.path.join(data_root, s) for s in shards]
        self.shards = shards
        assert len(shards) > 0, f'no shards found for split {split}'

        self.current_shard = 0
        self.tokens = load_tokens(self.shards[self.current_shard])
        self.current_position = 0

    def next_batch(self):
        B, T = self.B, self.T
        #as usual. gather B*T + 1 data. first B * T to input. shift input with one data point and make them target
        buf = self.tokens[self.current_position: self.current_position + (B * T) + 1] #taking batch + 1
        x = (buf[:-1]).view(B,T) #till last one
        y = (buf[1:]).view(B,T)  #second 2 last
        #advancing position in big token tensor
        self.current_position += B*T
        #end of shard → load next shard
        if self.current_position + (B * T + 1) > len(self.tokens):
            self.current_shard = (self.current_shard + 1) % len(self.shards)
            self.tokens = load_tokens(self.shards[self.current_shard])
            self.current_position = 0
        return x,y

In [5]:
local_dir = "/content/drive/MyDrive/yz50/edu_fineweb10B"
remote_name = "sample-10BT"
shard_size = int(1e7) #10m token per shard and 10 shards so 100m in total

In [6]:
#create the cache the local directory if it doesn't exist yet
DATA_CACHE_DIR = local_dir
os.makedirs(DATA_CACHE_DIR, exist_ok=True)

#stream the dataset
fw = load_dataset("HuggingFaceFW/fineweb-edu", name=remote_name, split="train",streaming=True)

#cinit the tokenizer
enc = tiktoken.get_encoding("gpt2")
eot = enc._special_tokens['<|endoftext|>'] #end of text token
def tokenize(doc):
    # tokenizes a single document and returns a numpy array of uint16 tokens
    tokens = [eot] #special end of text token
    tokens.extend(enc.encode_ordinary(doc["text"]))
    tokens_np = np.array(tokens)
    assert (0 <= tokens_np).all() and (tokens_np < 2**16).all(), "token dictionary too large for uint16"
    tokens_np_uint16 = tokens_np.astype(np.uint16)
    return tokens_np_uint16

def write_datafile(filename, tokens_np):
    np.save(filename, tokens_np)

#tokenize documents and write 10 shards of shard_size tokens
nprocs = max(1, os.cpu_count()//2)
with mp.Pool(nprocs) as pool:
    shard_index = 0
    # preallocate buffer to hold current shard
    all_tokens_np = np.empty((shard_size,), dtype=np.uint16)
    token_count = 0
    progress_bar = None
    for tokens in pool.imap(tokenize, fw, chunksize=16):

        #space checking for next tokens
        if token_count + len(tokens) < shard_size:
            #appending tokens to current shard
            all_tokens_np[token_count:token_count+len(tokens)] = tokens
            token_count += len(tokens)
            #bar update
            if progress_bar is None:
                progress_bar = tqdm(total=shard_size, unit="tokens", desc=f"Shard {shard_index}")
            progress_bar.update(len(tokens))
        else:
            #writing only 10 shards not whole shards
            split = "val" if shard_index == 0 else "train"
            filename = os.path.join(DATA_CACHE_DIR, f"edufineweb_{split}_{shard_index:06d}")
            #taking as much as it takes and sending remainder to next shard
            remainder = shard_size - token_count
            progress_bar.update(remainder)
            all_tokens_np[token_count:token_count+remainder] = tokens[:remainder]
            write_datafile(filename, all_tokens_np)
            shard_index += 1
            if shard_index == 10:
                break
            progress_bar = None
            # populate the next shard with the leftovers of the current doc
            all_tokens_np[0:len(tokens)-remainder] = tokens[remainder:]
            token_count = len(tokens)-remainder


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/26.4k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/2410 [00:00<?, ?it/s]

Shard 9:  99%|█████████▊| 9873895/10000000 [00:03<00:00, 2932994.83tokens/s]

In [7]:
train_loader = DataLoaderLite(B=B, T=T, split="train")
x, y = train_loader.next_batch()
print(x.shape, y.shape)
print(sorted(os.listdir(local_dir)))

torch.Size([16, 1024]) torch.Size([16, 1024])
['edufineweb_train_000001.npy', 'edufineweb_train_000002.npy', 'edufineweb_train_000003.npy', 'edufineweb_train_000004.npy', 'edufineweb_train_000005.npy', 'edufineweb_train_000006.npy', 'edufineweb_train_000007.npy', 'edufineweb_train_000008.npy', 'edufineweb_train_000009.npy', 'edufineweb_val_000000.npy']
